# English → Telugu Preference Optimization
### Resume from existing artifacts • Google Colab • Tesla T4

This notebook **does not regenerate the data pipeline**.

It reuses the files and the trained SFT LoRA adapter already present in:

`/content/drive/MyDrive/telugu-project/`

Existing artifacts used here:
- `telugu_candidates_raw.jsonl`
- `telugu_judged_raw.jsonl`
- `preference_pairs.jsonl`
- `telugu_preferences.jsonl`
- `telugu_train.jsonl`
- `telugu_heldout.jsonl`
- `sft_checkpoint/`
- `sft_adapter/`

The next stage is **DPO fine-tuning starting from the existing SFT adapter**, followed by translation/evaluation.

> **Important:** Do not delete or recreate `/content/drive`. The Drive mount cell below is the only mount cell in this notebook.


## 1. Install compatible packages

Use the pinned versions below. In particular, `trl==0.12.2` is used because this version's DPO API supports the encoder-decoder settings needed by IndicTrans2.


In [1]:
%pip install -q \
    "transformers==4.51.3" \
    "peft==0.15.2" \
    "accelerate==1.2.1" \
    "bitsandbytes>=0.46.0" \
    "indictranstoolkit==1.1.1" \
    "datasets>=3.2.0" \
    "sentencepiece" \
    "huggingface_hub" \
    "groq"

%pip install -q --no-deps "trl==0.12.2"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 66.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.1/411.1 kB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.4/336.4 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.4/548.4 kB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 60.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 867.8/867.8 kB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.0/129.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
import importlib.metadata as md

for pkg in [
    "transformers",
    "trl",
    "peft",
    "accelerate",
    "bitsandbytes",
    "indictranstoolkit",
    "huggingface-hub",
]:
    try:
        print(f"{pkg}: {md.version(pkg)}")
    except Exception as e:
        print(f"{pkg}: NOT INSTALLED")

transformers: 4.51.3
trl: 0.12.2
peft: 0.15.2
accelerate: 1.2.1
bitsandbytes: 0.50.2
indictranstoolkit: 1.1.1
huggingface-hub: 0.36.2


### ⚠️ Restart the Colab runtime once

After the installation cell finishes, use **Runtime → Restart session**.

After restarting, continue from **Section 2**. Do not rerun the installation cell repeatedly.


## 2. Imports and configuration


In [2]:
import os
import json
import random
import re
import time
import torch

PROJECT_PATH = "/content/drive/MyDrive/telugu-project/"
MODEL_NAME = "ai4bharat/indictrans2-en-indic-dist-200M"

DPO_BATCH_SIZE = 2
DPO_GRAD_ACCUMULATION = 4
DPO_EPOCHS = 2
DPO_LEARNING_RATE = 5e-5
DPO_BETA = 0.1

print("Project path:", PROJECT_PATH)


Project path: /content/drive/MyDrive/telugu-project/


In [3]:
from IndicTransToolkit import IndicProcessor

print("IndicTransToolkit: OK")

IndicTransToolkit: OK


## 3. Verify GPU and package versions


In [4]:
import torch
import transformers
import trl
import peft
from IndicTransToolkit import IndicProcessor

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Transformers:", transformers.__version__)
print("TRL:", trl.__version__)
print("PEFT:", peft.__version__)
print("IndicTransToolkit: OK")

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
Transformers: 4.51.3
TRL: 0.12.2
PEFT: 0.15.2
IndicTransToolkit: OK


## 4. Mount Google Drive — only once

This cell **does not create, delete, or rename** the project folder.

Do not use `rm -rf /content/drive`.


In [5]:
from google.colab import drive

drive.mount("/content/drive")

if not os.path.isdir(PROJECT_PATH):
    raise FileNotFoundError(
        f"Project folder was not found: {PROJECT_PATH}"
    )

print("Drive mounted.")
print("Project folder:", PROJECT_PATH)


Mounted at /content/drive
Drive mounted.
Project folder: /content/drive/MyDrive/telugu-project/


## 5. Verify existing project files

This is a **read-only check**. Nothing is regenerated or overwritten.


In [6]:
required_files = [
    "telugu_candidates_raw.jsonl",
    "telugu_judged_raw.jsonl",
    "preference_pairs.jsonl",
    "telugu_preferences.jsonl",
    "telugu_train.jsonl",
    "telugu_heldout.jsonl",
]

required_dirs = [
    "sft_checkpoint",
    "sft_adapter",
]

missing = []

for name in required_files:
    path = os.path.join(PROJECT_PATH, name)
    if os.path.isfile(path):
        print(f"✅ {name}")
    else:
        print(f"❌ MISSING: {name}")
        missing.append(name)

for name in required_dirs:
    path = os.path.join(PROJECT_PATH, name)
    if os.path.isdir(path):
        print(f"✅ {name}/")
    else:
        print(f"❌ MISSING: {name}/")
        missing.append(name)

if missing:
    raise FileNotFoundError(
        "Required existing project artifacts are missing: "
        + ", ".join(missing)
    )

print("\nAll required existing artifacts are available.")


✅ telugu_candidates_raw.jsonl
✅ telugu_judged_raw.jsonl
✅ preference_pairs.jsonl
✅ telugu_preferences.jsonl
✅ telugu_train.jsonl
✅ telugu_heldout.jsonl
✅ sft_checkpoint/
✅ sft_adapter/

All required existing artifacts are available.


## 6. Hugging Face authentication


In [7]:
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN was not found. Add it to Colab Secrets."
    )

login(token=HF_TOKEN)
print("✅ Hugging Face authentication ready.")


✅ Hugging Face authentication ready.


## 7. Load IndicTrans2 and IndicTransProcessor


In [8]:
from IndicTransToolkit import IndicProcessor
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, BitsAndBytesConfig

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    trust_remote_code=True
)

ip = IndicProcessor(inference=False)

print("Tokenizer and IndicProcessor loaded.")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenization_indictrans.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-en-indic-dist-200M:
- tokenization_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


dict.SRC.json: 0.00B [00:00, ?B/s]

dict.TGT.json: 0.00B [00:00, ?B/s]

model.SRC:   0%|          | 0.00/759k [00:00<?, ?B/s]

model.TGT:   0%|          | 0.00/3.26M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/96.0 [00:00<?, ?B/s]

Tokenizer and IndicProcessor loaded.


## 8. Load the existing SFT adapter

We are **not training SFT again**.

The existing `sft_adapter/` is loaded on the original IndicTrans2 base model in 4-bit form. DPO will continue from this SFT policy.


In [16]:
import transformers, peft, trl

print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)

Transformers: 4.51.3
PEFT: 0.15.2
TRL: 0.12.2


In [ ]:
import transformers
print("Transformers:", transformers.__version__)

In [19]:
from peft import PeftModel
from transformers import AutoModelForSeq2SeqLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map="auto",
)

model = PeftModel.from_pretrained(
    base_model,
    os.path.join(PROJECT_PATH, "sft_adapter"),
    is_trainable=True,
    adapter_name="dpo_train",
)

model.load_adapter(
    os.path.join(PROJECT_PATH, "sft_adapter"),
    adapter_name="dpo_reference",
)

model.set_adapter("dpo_train")

print("✅ Base model + existing SFT adapter loaded.")
print("Trainable adapter: dpo_train")
print("Reference adapter: dpo_reference")

✅ Base model + existing SFT adapter loaded.
Trainable adapter: dpo_train
Reference adapter: dpo_reference


## 9. Load the existing DPO preference dataset

The existing `telugu_train.jsonl` already contains:

`prompt`, `chosen`, `rejected`

We only preprocess the English prompt with the same IndicTrans2 processor used by the earlier SFT stage.

**No new candidate generation, Groq judging, manual review, or train/held-out split is performed.**


In [20]:
from datasets import load_dataset

train_data = load_dataset(
    "json",
    data_files=os.path.join(PROJECT_PATH, "telugu_train.jsonl")
)["train"]

heldout_data = load_dataset(
    "json",
    data_files=os.path.join(PROJECT_PATH, "telugu_heldout.jsonl")
)["train"]

print("Training pairs:", len(train_data))
print("Held-out pairs:", len(heldout_data))
print("\nExisting example:")
print(train_data[0])


Training pairs: 339
Held-out pairs: 60

Existing example:
{'prompt': 'Translate to Telugu: Vallimalai can boast of being home to three temples', 'chosen': 'వల్లిమలై మూడు దేవాలయాలకు నిలయం అని గర్వగించవచ్చు.', 'rejected': 'వల్లిమలై మూడు దేవాలయాలకు నిలయం అని ప్రగల్భాలు పలుకుతుంది...............................................................................................................................................................................................................................................'}


## 10. Prepare preference data for IndicTrans2


In [21]:
def extract_english(prompt):
    prefix = "Translate to Telugu: "
    return prompt[len(prefix):] if prompt.startswith(prefix) else prompt

def prepare_dpo_examples(examples):
    english = [extract_english(p) for p in examples["prompt"]]

    processed_source = ip.preprocess_batch(
        english,
        src_lang="eng_Latn",
        tgt_lang="tel_Telu"
    )

    return {
        "prompt": processed_source,
        "chosen": examples["chosen"],
        "rejected": examples["rejected"],
    }

dpo_train = train_data.map(
    prepare_dpo_examples,
    batched=True,
    remove_columns=train_data.column_names,
)

print(dpo_train)
print("\nPrepared example:")
print(dpo_train[0])


Map:   0%|          | 0/339 [00:00<?, ? examples/s]

Dataset({
    features: ['prompt', 'chosen', 'rejected'],
    num_rows: 339
})

Prepared example:
{'prompt': 'eng_Latn tel_Telu Vallimalai can boast of being home to three temples', 'chosen': 'వల్లిమలై మూడు దేవాలయాలకు నిలయం అని గర్వగించవచ్చు.', 'rejected': 'వల్లిమలై మూడు దేవాలయాలకు నిలయం అని ప్రగల్భాలు పలుకుతుంది...............................................................................................................................................................................................................................................'}


In [26]:
# Check IndicTrans2's separate source/target tokenization modes

test_prompt = dpo_train[0]["prompt"]
test_target = dpo_train[0]["chosen"]

print("PROMPT:")
print(test_prompt)

print("\nTARGET:")
print(test_target)

tokenizer._switch_to_input_mode()
src_ids = tokenizer(test_prompt, add_special_tokens=False)["input_ids"]

tokenizer._switch_to_target_mode()
tgt_ids = tokenizer(test_target, add_special_tokens=False)["input_ids"]

tokenizer._switch_to_input_mode()

print("\nSource token IDs:", src_ids[:20])
print("Target token IDs:", tgt_ids[:20])
print("Target token count:", len(tgt_ids))

PROMPT:
eng_Latn tel_Telu Vallimalai can boast of being home to three temples

TARGET:
వల్లిమలై మూడు దేవాలయాలకు నిలయం అని గర్వగించవచ్చు.


AssertionError: Invalid source language tag: వల్లిమలై

In [27]:
# Directly test the target SentencePiece tokenizer.
# This bypasses IndicTransTokenizer.__call__ completely.

test_target = dpo_train[0]["chosen"]

target_pieces = tokenizer.tgt_spm.EncodeAsPieces(test_target)

print("Target text:")
print(test_target)

print("\nTarget pieces:")
print(target_pieces[:30])

print("\nNumber of target pieces:")
print(len(target_pieces))

Target text:
వల్లిమలై మూడు దేవాలయాలకు నిలయం అని గర్వగించవచ్చు.

Target pieces:
['▁', 'వ', 'ల', '్', 'ల', 'ి', 'మ', 'ల', 'ై', '▁', 'మ', 'ూ', 'డ', 'ు', '▁', 'ద', 'ే', 'వ', 'ా', 'ల', 'య', 'ా', 'ల', 'క', 'ు', '▁', 'న', 'ి', 'ల', 'య']

Number of target pieces:
50


## 11. Configure DPO

For IndicTrans2, this is an **encoder-decoder** DPO setup.

The DPO reference policy is the second copy of the existing SFT adapter, so DPO learns to improve the SFT policy relative to the same SFT starting point.


In [32]:
from trl import DPOTrainer, DPOConfig


class IndicTransDPOTrainer(DPOTrainer):

    # ----------------------------------------------------------
    # Fix 1:
    # IndicTrans2 requires different tokenization for
    # source (English) and target (Telugu).
    # ----------------------------------------------------------
    @staticmethod
    def tokenize_row(
        features,
        processing_class,
        max_prompt_length,
        max_completion_length,
        add_special_tokens,
    ):
        tokenizer = processing_class

        # English/source
        prompt_input_ids = tokenizer(
            features["prompt"],
            add_special_tokens=False,
        )["input_ids"]

        # Telugu/target
        # Bypass IndicTransTokenizer.__call__()
        # because it incorrectly treats Telugu as source text.
        chosen_pieces = tokenizer.tgt_spm.EncodeAsPieces(
            features["chosen"]
        )

        rejected_pieces = tokenizer.tgt_spm.EncodeAsPieces(
            features["rejected"]
        )

        chosen_input_ids = tokenizer.convert_tokens_to_ids(
            chosen_pieces
        )

        rejected_input_ids = tokenizer.convert_tokens_to_ids(
            rejected_pieces
        )

        # EOS
        if tokenizer.eos_token_id is not None:
            chosen_input_ids.append(tokenizer.eos_token_id)
            rejected_input_ids.append(tokenizer.eos_token_id)

        # Truncate
        if max_prompt_length is not None:
            prompt_input_ids = prompt_input_ids[-max_prompt_length:]

        if max_completion_length is not None:
            chosen_input_ids = chosen_input_ids[:max_completion_length]
            rejected_input_ids = rejected_input_ids[:max_completion_length]

        # TRL 0.12.2 expects these exact names
        return {
            "prompt_input_ids": prompt_input_ids,
            "chosen_input_ids": chosen_input_ids,
            "rejected_input_ids": rejected_input_ids,
        }

    # ----------------------------------------------------------
    # Fix 2:
    # TRL 0.12.2's DPOTrainer.log() is incompatible with
    # Transformers 4.51.3, which passes start_time.
    # ----------------------------------------------------------
    def log(self, logs, start_time=None):
        return super().log(logs)


# --------------------------------------------------------------
# DPO CONFIG
# --------------------------------------------------------------

dpo_config = DPOConfig(
    output_dir=os.path.join(PROJECT_PATH, "dpo_checkpoint"),

    per_device_train_batch_size=DPO_BATCH_SIZE,
    gradient_accumulation_steps=DPO_GRAD_ACCUMULATION,

    num_train_epochs=DPO_EPOCHS,
    learning_rate=DPO_LEARNING_RATE,
    beta=DPO_BETA,

    logging_steps=10,
    save_strategy="epoch",

    fp16=True,

    is_encoder_decoder=True,

    max_length=256,
    max_prompt_length=128,
    max_target_length=128,

    report_to="none",
)


# --------------------------------------------------------------
# DPO TRAINER
# --------------------------------------------------------------

dpo_trainer = IndicTransDPOTrainer(
    model=model,
    ref_model=None,
    args=dpo_config,
    train_dataset=dpo_train,
    processing_class=tokenizer,
)

print("✅ DPOTrainer ready.")

/usr/local/lib/python3.13/dist-packages/trl/trainer/dpo_config.py:182: FutureWarning: The `max_target_length` argument is deprecated in favor of `max_completion_length` and will be removed in a future version.
  warnings.warn(


Tokenizing train dataset:   0%|          | 0/339 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForSeq2SeqLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


✅ DPOTrainer ready.


## 12. Train DPO

This is the first cell in this notebook that **changes model weights**.

It does not regenerate the preference data.

If training is interrupted after a checkpoint is saved, the checkpoint can be resumed instead of starting from zero.


In [33]:
dpo_trainer.train()


Step,Training Loss
10,1.580500
20,1.791900
30,1.639600
40,1.133800
50,1.208300
60,1.240800
70,1.657900
80,1.219700


TrainOutput(global_step=84, training_loss=1.4187796456473214, metrics={'train_runtime': 235.0977, 'train_samples_per_second': 2.884, 'train_steps_per_second': 0.357, 'total_flos': 0.0, 'train_loss': 1.4187796456473214, 'epoch': 1.9647058823529413})

## 13. Save the DPO adapter


In [34]:
dpo_adapter_path = os.path.join(PROJECT_PATH, "dpo_adapter")

model.save_pretrained(dpo_adapter_path)
tokenizer.save_pretrained(dpo_adapter_path)

print("✅ DPO adapter saved to:")
print(dpo_adapter_path)


✅ DPO adapter saved to:
/content/drive/MyDrive/telugu-project/dpo_adapter


## 14. Test the trained DPO model


In [40]:
# ============================================================
# SECTION 14 — DPO TEST TRANSLATION
# ============================================================

from IndicTransToolkit import IndicProcessor

# Fresh processor ONLY for inference
inference_ip = IndicProcessor(inference=True)

test_sentence = "How are you doing today?"

# Preprocess and postprocess must use the SAME processor
src = inference_ip.preprocess_batch(
    [test_sentence],
    src_lang="eng_Latn",
    tgt_lang="tel_Telu"
)

print("Processed source:", src)

inputs = tokenizer(
    src,
    return_tensors="pt",
    truncation=True,
    max_length=128
)

inputs = {
    k: v.to(model.device)
    for k, v in inputs.items()
}

with torch.inference_mode():
    output = model.generate(
        **inputs,
        num_beams=1,
        max_new_tokens=64,
        use_cache=True,
    )

print("Generation finished.")

tokenizer._switch_to_target_mode()

decoded = tokenizer.batch_decode(
    output,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=True
)

tokenizer._switch_to_input_mode()

print("Decoded:", decoded)

print("Starting postprocess...")

result = inference_ip.postprocess_batch(
    decoded,
    lang="tel_Telu"
)

print("Postprocess finished.")
print("DPO Telugu:", result[0])

Processed source: ['eng_Latn tel_Telu How are you doing today ?']
Generation finished.
Decoded: ['ई रोजु मीरु ऎला उन्नारु?']
Starting postprocess...
Postprocess finished.
DPO Telugu: ఈ రోజు మీరు ఎలా ఉన్నారు?


## 15. Load the existing SFT adapter fresh, for SimPO

SimPO starts from the same SFT policy as DPO, but does **not** need a frozen reference model copy — it uses the model's own length-normalized log-probability as the reward signal instead.

In [42]:
from peft import PeftModel

SIMPO_BATCH_SIZE = 2
SIMPO_GRAD_ACCUMULATION = 4
SIMPO_EPOCHS = 2
SIMPO_LEARNING_RATE = 5e-5
SIMPO_BETA = 2.0
SIMPO_GAMMA = 0.5

simpo_base_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map="auto",
)

simpo_model = PeftModel.from_pretrained(
    simpo_base_model,
    os.path.join(PROJECT_PATH, "sft_adapter"),
    is_trainable=True,
    adapter_name="simpo_train",
)

simpo_model.set_adapter("simpo_train")

print("Base model + existing SFT adapter loaded for SimPO.")

Base model + existing SFT adapter loaded for SimPO.


## 16. Configure and run SimPO training

Reuses `dpo_train` (already preprocessed with the IndicTrans2 source format in Section 10) - the same `{prompt, chosen, rejected}` data works for all three preference methods.

In [49]:
from trl import CPOTrainer, CPOConfig


class IndicTransCPOTrainer(CPOTrainer):

    def tokenize_row(
        self,
        feature,
        model=None,
    ):
        prompt = feature["prompt"]
        chosen = feature["chosen"]
        rejected = feature["rejected"]

        # Source / prompt
        prompt_tokens = self.processing_class(
            prompt,
            truncation=True,
            max_length=self.max_prompt_length,
            add_special_tokens=True,
        )

        # Telugu target tokenizer
        def tokenize_target(text):
            pieces = self.processing_class.tgt_spm.EncodeAsPieces(text)

            ids = [
                self.processing_class.tgt_encoder.get(
                    piece,
                    self.processing_class.unk_token_id,
                )
                for piece in pieces
            ]

            ids = ids[: self.max_completion_length]

            if (
                self.processing_class.eos_token_id is not None
                and (
                    len(ids) == 0
                    or ids[-1] != self.processing_class.eos_token_id
                )
            ):
                ids.append(self.processing_class.eos_token_id)

            return ids

        chosen_ids = tokenize_target(chosen)
        rejected_ids = tokenize_target(rejected)

        return {
            "prompt_input_ids": prompt_tokens["input_ids"],
            "prompt_attention_mask": prompt_tokens["attention_mask"],

            "chosen_input_ids": chosen_ids,
            "chosen_labels": chosen_ids.copy(),

            "rejected_input_ids": rejected_ids,
            "rejected_labels": rejected_ids.copy(),
        }

    # Compatibility fix for TRL 0.12.2 + Transformers 4.51.3
    def log(self, logs, start_time=None):
        return super().log(logs)


simpo_config = CPOConfig(
    output_dir=os.path.join(PROJECT_PATH, "simpo_checkpoint"),
    loss_type="simpo",
    per_device_train_batch_size=SIMPO_BATCH_SIZE,
    gradient_accumulation_steps=SIMPO_GRAD_ACCUMULATION,
    num_train_epochs=SIMPO_EPOCHS,
    learning_rate=SIMPO_LEARNING_RATE,
    beta=SIMPO_BETA,
    cpo_alpha=0.0,
    simpo_gamma=SIMPO_GAMMA,
    logging_steps=10,
    save_strategy="epoch",
    fp16=True,
    is_encoder_decoder=True,
    max_length=256,
    max_prompt_length=128,
    max_completion_length=128,
    remove_unused_columns=False,
    report_to="none",
)


simpo_trainer = IndicTransCPOTrainer(
    model=simpo_model,
    args=simpo_config,
    train_dataset=dpo_train,
    processing_class=tokenizer,
)

print("CPOTrainer (SimPO) ready.")

Map:   0%|          | 0/339 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForSeq2SeqLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


CPOTrainer (SimPO) ready.


## 17. Train SimPO

This changes model weights. No preference data is regenerated.

In [50]:
simpo_trainer.train()


Step,Training Loss
10,1.245600
20,1.070600
30,1.064400
40,1.076900
50,0.962800
60,0.973800
70,1.012900
80,0.978600


TrainOutput(global_step=84, training_loss=1.0453092455863953, metrics={'train_runtime': 151.987, 'train_samples_per_second': 4.461, 'train_steps_per_second': 0.553, 'total_flos': 0.0, 'train_loss': 1.0453092455863953, 'epoch': 1.9647058823529413})

## 18. Save the SimPO adapter

In [65]:
simpo_adapter_path = os.path.join(
    PROJECT_PATH,
    "simpo_adapter"
)

simpo_model.save_pretrained(simpo_adapter_path)
tokenizer.save_pretrained(simpo_adapter_path)

print("SimPO adapter saved to:")
print(simpo_adapter_path)

SimPO adapter saved to:
/content/drive/MyDrive/telugu-project/simpo_adapter


## 19. Test the trained SimPO model

In [53]:
# ============================================================
# SECTION 19 — Test the trained SimPO model
# ============================================================

from IndicTransToolkit import IndicProcessor

simpo_inference_ip = IndicProcessor(inference=True)

print("English:", test_sentence)

src = simpo_inference_ip.preprocess_batch(
    [test_sentence],
    src_lang="eng_Latn",
    tgt_lang="tel_Telu"
)

inputs = tokenizer(
    src,
    return_tensors="pt",
    truncation=True,
    max_length=128
)

inputs = {
    k: v.to(simpo_model.device)
    for k, v in inputs.items()
}

with torch.inference_mode():
    output = simpo_model.generate(
        **inputs,
        num_beams=1,
        max_new_tokens=64,
        use_cache=True,
    )

print("Generation finished.")

tokenizer._switch_to_target_mode()

decoded = tokenizer.batch_decode(
    output,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=True
)

tokenizer._switch_to_input_mode()

print("Decoded:", decoded)

result = simpo_inference_ip.postprocess_batch(
    decoded,
    lang="tel_Telu"
)

print("SimPO Telugu:", result[0])

English: How are you doing today?
Generation finished.
Decoded: ['ई रोजु मीरु ऎला उन्नारु?']
SimPO Telugu: ఈ రోజు మీరు ఎలా ఉన్నారు?


## 20. Prepare ORPO - starts from the raw base model, not the SFT adapter

ORPO combines imitation learning and preference alignment into a single training stage using an odds-ratio penalty term, so it does not build on top of the existing SFT policy the way DPO and SimPO do. A fresh LoRA adapter is attached to the unmodified base model.

In [54]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

ORPO_BATCH_SIZE = 2
ORPO_GRAD_ACCUMULATION = 4
ORPO_EPOCHS = 3
ORPO_LEARNING_RATE = 8e-6
ORPO_BETA = 0.1

orpo_base_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map="auto",
)
orpo_base_model = prepare_model_for_kbit_training(orpo_base_model)

# Uses the same target_modules the original SFT LoRA config was built on (Section 7/8).
# If this raises "no LoRA layers found" or trainable params print as 0,
# run: for n, m in orpo_base_model.named_modules(): print(n)
# and correct target_modules to match the real layer names before re-running.
orpo_lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "out_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_2_SEQ_LM",
)

orpo_model = get_peft_model(orpo_base_model, orpo_lora_config)
orpo_model.print_trainable_parameters()


You are using an old version of the checkpointing format that is deprecated (We will also silently ignore `gradient_checkpointing_kwargs` in case you passed it).Please update to the new format on your modeling file. To use the new format, you need to completely remove the definition of the method `_set_gradient_checkpointing` in your model.


trainable params: 3,538,944 || all params: 215,315,456 || trainable%: 1.6436


## 21. Configure and run ORPO training

In [57]:
# ============================================================
# SECTION 21 — Configure and run ORPO training
# ============================================================

from trl import ORPOTrainer, ORPOConfig


class IndicTransORPOTrainer(ORPOTrainer):

    def tokenize_row(
        self,
        feature,
        model=None,
    ):
        prompt = feature["prompt"]
        chosen = feature["chosen"]
        rejected = feature["rejected"]

        # Tokenize English source prompt normally
        prompt_tokens = self.processing_class(
            prompt,
            truncation=True,
            max_length=self.max_prompt_length,
            add_special_tokens=True,
        )

        # Tokenize Telugu targets with TARGET SentencePiece
        def tokenize_target(text):
            pieces = self.processing_class.tgt_spm.EncodeAsPieces(text)

            ids = [
                self.processing_class.tgt_encoder.get(
                    piece,
                    self.processing_class.unk_token_id,
                )
                for piece in pieces
            ]

            ids = ids[:self.max_completion_length]

            if (
                self.processing_class.eos_token_id is not None
                and (
                    len(ids) == 0
                    or ids[-1] != self.processing_class.eos_token_id
                )
            ):
                ids.append(self.processing_class.eos_token_id)

            return ids

        chosen_ids = tokenize_target(chosen)
        rejected_ids = tokenize_target(rejected)

        return {
            "prompt_input_ids": prompt_tokens["input_ids"],
            "prompt_attention_mask": prompt_tokens["attention_mask"],

            "chosen_input_ids": chosen_ids,
            "chosen_labels": chosen_ids.copy(),

            "rejected_input_ids": rejected_ids,
            "rejected_labels": rejected_ids.copy(),
        }

    # Compatibility fix for TRL 0.12.2 + Transformers 4.51.3
    def log(self, logs, start_time=None):
        return super().log(logs)


orpo_config = ORPOConfig(
    output_dir=os.path.join(PROJECT_PATH, "orpo_checkpoint"),
    per_device_train_batch_size=ORPO_BATCH_SIZE,
    gradient_accumulation_steps=ORPO_GRAD_ACCUMULATION,
    num_train_epochs=ORPO_EPOCHS,
    learning_rate=ORPO_LEARNING_RATE,
    beta=ORPO_BETA,
    logging_steps=10,
    save_strategy="epoch",
    fp16=True,
    is_encoder_decoder=True,
    max_length=256,
    max_prompt_length=128,
    max_completion_length=128,
    remove_unused_columns=False,
    report_to="none",
)


orpo_trainer = IndicTransORPOTrainer(
    model=orpo_model,
    args=orpo_config,
    train_dataset=dpo_train,
    processing_class=tokenizer,
)

print("ORPOTrainer ready.")

Map:   0%|          | 0/339 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForSeq2SeqLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


ORPOTrainer ready.


In [60]:
orpo_trainer.train()

/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,10.781800
20,10.626500
30,10.665600
40,10.627000
50,10.145700
60,10.582500
70,10.712200
80,10.445800
90,9.970100
100,10.635600


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


TrainOutput(global_step=126, training_loss=10.51690307496086, metrics={'train_runtime': 339.963, 'train_samples_per_second': 2.992, 'train_steps_per_second': 0.371, 'total_flos': 0.0, 'train_loss': 10.51690307496086, 'epoch': 2.9411764705882355})

## 22. Save the ORPO adapter

In [61]:
orpo_adapter_path = os.path.join(PROJECT_PATH, "orpo_adapter")

orpo_model.save_pretrained(orpo_adapter_path)
tokenizer.save_pretrained(orpo_adapter_path)

print("ORPO adapter saved to:")
print(orpo_adapter_path)


ORPO adapter saved to:
/content/drive/MyDrive/telugu-project/orpo_adapter


## 23. Test the trained ORPO model

In [63]:
# ============================================================
# SECTION 23 — Test the trained ORPO model
# ============================================================

from IndicTransToolkit import IndicProcessor

orpo_inference_ip = IndicProcessor(inference=True)

print("English:", test_sentence)

src = orpo_inference_ip.preprocess_batch(
    [test_sentence],
    src_lang="eng_Latn",
    tgt_lang="tel_Telu"
)

inputs = tokenizer(
    src,
    return_tensors="pt",
    truncation=True,
    max_length=128
)

inputs = {
    k: v.to(orpo_model.device)
    for k, v in inputs.items()
}

with torch.inference_mode():
    output = orpo_model.generate(
        **inputs,
        num_beams=1,
        max_new_tokens=64,
        use_cache=True,
    )

print("Generation finished.")

tokenizer._switch_to_target_mode()

decoded = tokenizer.batch_decode(
    output,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=True
)

tokenizer._switch_to_input_mode()

print("Decoded:", decoded)

result = orpo_inference_ip.postprocess_batch(
    decoded,
    lang="tel_Telu"
)

print("ORPO Telugu:", result[0])

English: How are you doing today?
Generation finished.
Decoded: ['ई रोजु ऎला उन्नारु?']
ORPO Telugu: ఈ రోజు ఎలా ఉన్నారు?


## 24. Load all four models for evaluation (SFT, DPO, SimPO, ORPO)

Each adapter is loaded on its own fresh base model instance to avoid any adapter-switching state bleeding between models during evaluation.

In [66]:
from peft import PeftModel

def load_eval_model(adapter_subfolder, adapter_name=None):

    eval_base = AutoModelForSeq2SeqLM.from_pretrained(
        MODEL_NAME,
        token=HF_TOKEN,
        trust_remote_code=True,
        quantization_config=bnb_config,
        device_map="auto",
    )

    adapter_path = os.path.join(
        PROJECT_PATH,
        adapter_subfolder
    )

    if adapter_name is not None:
        adapter_path = os.path.join(
            adapter_path,
            adapter_name
        )

    return PeftModel.from_pretrained(
        eval_base,
        adapter_path,
    )


eval_models = {
    "sft": load_eval_model("sft_adapter"),
    "dpo": load_eval_model("dpo_adapter"),
    "simpo": load_eval_model(
        "simpo_adapter",
        "simpo_train"
    ),
    "orpo": load_eval_model("orpo_adapter"),
}

print(
    "All four models loaded for evaluation:",
    list(eval_models.keys())
)

All four models loaded for evaluation: ['sft', 'dpo', 'simpo', 'orpo']


## 25. Generate translations from all four models on the held-out set

Uses the existing `telugu_heldout.jsonl` - no new data is created here.

In [68]:
# ============================================================
# SECTION 25 — Generate translations from all four models
# ============================================================

eval_results = []

eval_ip = IndicProcessor(inference=True)


def translate_eval(model, sentence):
    src = eval_ip.preprocess_batch(
        [sentence],
        src_lang="eng_Latn",
        tgt_lang="tel_Telu"
    )

    inputs = tokenizer(
        src,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )

    inputs = {
        k: v.to(model.device)
        for k, v in inputs.items()
    }

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            num_beams=1,
            max_new_tokens=128,
            use_cache=True,
        )

    tokenizer._switch_to_target_mode()

    decoded = tokenizer.batch_decode(
        output,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True
    )

    tokenizer._switch_to_input_mode()

    return eval_ip.postprocess_batch(
        decoded,
        lang="tel_Telu"
    )[0]


for item in heldout_data:

    english = extract_english(item["prompt"])

    row = {
        "english": english,
        "reference": item.get("reference_telugu", "")
    }

    for name, m in eval_models.items():
        row[name] = translate_eval(m, english)

    eval_results.append(row)

    print(f"\nEN:    {english}")
    for name in eval_models:
        print(f"{name.upper():6s}: {row[name]}")

print(
    f"\nEvaluated {len(eval_results)} held-out examples "
    f"across {len(eval_models)} models."
)


EN:    The Maestro Edge 125 is also equipped with Hero MotoCorp's i3S (idle-Start-Stop-System) technology, which helps improve the fuel efficiency
SFT   : మాస్ట్రో ఎడ్జ్ 125 లో హీరో మోటోకార్ప్ యొక్క ఐ3ఎస్ (ఐడిల్-స్టార్ట్-స్టాప్-సిస్టమ్) టెక్నాలజీ కూడా ఉంది, ఇది ఇంధన సామర్థ్యాన్ని మెరుగుపరచడంలో సహాయపడుతుంది
DPO   : మాస్ట్రో ఎడ్జ్ 125 లో హీరో మోటోకార్ప్ యొక్క ఐ3ఎస్ (ఐడిల్-స్టార్ట్-స్టాప్-సిస్టమ్) టెక్నాలజీ కూడా ఉంది, ఇది ఇంధన సామర్థ్యాన్ని మెరుగుపరచడంలో సహాయపడుతుంది
SIMPO : మాస్ట్రో ఎడ్జ్ 125లో హీరో మోటోకార్ప్ యొక్క ఐ3ఎస్ (ఐడిల్-స్టార్ట్-స్టాప్-సిస్టమ్) టెక్నాలజీ కూడా ఉంది, ఇది ఇంధన సామర్థ్యాన్ని మెరుగుపరచడంలో సహాయపడుతుంది
ORPO  : మాస్ట్రో ఎడ్జ్ 125లో హీరో మోటోకార్ప్ యొక్క ఐ3ఎస్ (ఐడిల్-స్టార్ట్-స్టాప్-సిస్టమ్) టెక్నాలజీ కూడా ఉంది, ఇది ఇంధన సామర్థ్యాన్ని మెరుగుపరచడంలో సహాయపడుతుంది

EN:    is the question plaguing many.
SFT   : ఇది చాలా మందిని వేధిస్తున్న ప్రశ్న.
DPO   : ఇది చాలా మందిని వేధిస్తున్న ప్రశ్న.
SIMPO : ఇది చాలా మందిని వేధిస్తున్న ప్రశ్న.
ORPO  : అనేది చాలా మందిని వేధిస్తున్న ప

## 26. Pairwise judge tournament (Groq) - decides the overall winner

Every model is compared against every other model on every held-out prompt (6 matchups per prompt). Win counts across all matchups determine which alignment method actually performed best.

The Groq key is read from Colab Secrets, not hardcoded, to avoid exposing it if this notebook is shared.

In [75]:
# ============================================================
# SECTION 26 — GROQ TOURNAMENT (RESUME-SAFE + AUTO-SAVE)
# ============================================================

from groq import Groq, RateLimitError
from itertools import combinations
import os
import json
import re
import time

# ------------------------------------------------------------
# 1. GROQ CLIENT
# ------------------------------------------------------------

GROQ_API_KEY = userdata.get("API_KEY")

if not GROQ_API_KEY:
    raise RuntimeError("GROQ_API_KEY was not found. Add it to Colab Secrets.")

client = Groq(api_key=GROQ_API_KEY)

# ------------------------------------------------------------
# 2. TOURNAMENT PROMPT
# ------------------------------------------------------------

def build_tournament_prompt(english, out_1, out_2):
    template = (
        "You are evaluating two Telugu translations of an English sentence.\n\n"
        "English: {english}\n"
        "Translation 1: {out_1}\n"
        "Translation 2: {out_2}\n\n"
        "Judge strictly on:\n"
        "1. Accuracy - does it preserve the original meaning without adding or dropping information?\n"
        "2. Naturalness - does it read like something a native Telugu speaker would actually write?\n"
        "3. Grammar - correct Telugu grammar and script usage.\n\n"
        "Respond in this exact format:\n"
        "Winner: 1 or 2\n"
        "Reason: one sentence explaining why"
    )
    return template.format(
        english=english,
        out_1=out_1,
        out_2=out_2
    )


def judge_pair(english, out_1, out_2):

    prompt = build_tournament_prompt(
        english,
        out_1,
        out_2
    )

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.1,
    )

    text = response.choices[0].message.content

    match = re.search(
        r"winner\s*:\s*\**\s*([12])\b",
        text,
        re.IGNORECASE
    )

    winner_slot = match.group(1) if match else None

    return winner_slot, text


# ------------------------------------------------------------
# 3. FILE FOR PERMANENT CHECKPOINT
# ------------------------------------------------------------

checkpoint_file = os.path.join(
    PROJECT_PATH,
    "tournament_checkpoint.json"
)

print("Checkpoint file:")
print(checkpoint_file)


# ------------------------------------------------------------
# 4. LOAD EXISTING RESULTS
# ------------------------------------------------------------

existing_log = []

# First use results already present in this runtime
if "tournament_log" in globals():
    existing_log = list(tournament_log)

# Also load checkpoint from Drive if it exists
if os.path.exists(checkpoint_file):

    print("\nLoading checkpoint from Drive...")

    with open(checkpoint_file, "r", encoding="utf-8") as f:
        checkpoint_data = json.load(f)

    checkpoint_log = checkpoint_data.get(
        "tournament_log",
        []
    )

    # Combine checkpoint + current runtime results
    existing_log = checkpoint_log + existing_log

    print(
        f"Checkpoint contains "
        f"{len(checkpoint_log)} completed judgments."
    )


# ------------------------------------------------------------
# 5. REMOVE DUPLICATE MATCHUPS
# ------------------------------------------------------------

tournament_log = []

seen_keys = set()

for entry in existing_log:

    key = (
        entry.get("english"),
        entry.get("model_1"),
        entry.get("model_2"),
    )

    if key not in seen_keys:

        tournament_log.append(entry)
        seen_keys.add(key)


# ------------------------------------------------------------
# 6. MODELS + PAIRS
# ------------------------------------------------------------

model_names = list(eval_models.keys())

pairs = list(
    combinations(model_names, 2)
)

total_prompts = len(eval_results)

total_matchups = (
    total_prompts * len(pairs)
)


# ------------------------------------------------------------
# 7. REBUILD COUNTS FROM SAVED LOG
# ------------------------------------------------------------

win_counts = {
    name: 0
    for name in model_names
}

unparsed_count = 0

for entry in tournament_log:

    winner = entry.get("winner")

    if winner in win_counts:
        win_counts[winner] += 1
    else:
        unparsed_count += 1


# ------------------------------------------------------------
# 8. COMPLETED MATCHUPS
# ------------------------------------------------------------

completed_keys = {
    (
        entry.get("english"),
        entry.get("model_1"),
        entry.get("model_2"),
    )
    for entry in tournament_log
}


completed = len(completed_keys)

print()
print("=" * 60)
print("GROQ TOURNAMENT")
print("=" * 60)

print(f"Models: {model_names}")
print(f"Pairs per example: {len(pairs)}")
print(f"Held-out examples: {total_prompts}")
print(f"Total matchups: {total_matchups}")
print(f"Already completed: {completed}")
print(f"Remaining: {total_matchups - completed}")
print("=" * 60)


# ------------------------------------------------------------
# 9. SAVE FUNCTION
# ------------------------------------------------------------

def save_tournament_checkpoint():

    checkpoint_data = {
        "win_counts": win_counts,
        "unparsed_count": unparsed_count,
        "tournament_log": tournament_log,
        "heldout_translations": eval_results,
    }

    with open(
        checkpoint_file,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            checkpoint_data,
            f,
            ensure_ascii=False,
            indent=2
        )


# ------------------------------------------------------------
# 10. RUN ONLY MISSING MATCHUPS
# ------------------------------------------------------------

for row_idx, row in enumerate(eval_results):

    english = row["english"]

    print(
        f"\nExample {row_idx + 1}/{total_prompts}"
    )

    for m1, m2 in pairs:

        key = (
            english,
            m1,
            m2,
        )

        # --------------------------------------------
        # SKIP ALREADY COMPLETED MATCHUPS
        # --------------------------------------------

        if key in completed_keys:

            print(
                f"  SKIP: {m1} vs {m2} "
                f"(already saved)"
            )

            continue


        # --------------------------------------------
        # GROQ REQUEST WITH RETRY
        # --------------------------------------------

        while True:

            try:

                winner_slot, reason = judge_pair(
                    english,
                    row[m1],
                    row[m2]
                )

                break


            except RateLimitError as e:

                message = str(e)

                # Try to extract Groq's suggested wait time
                match = re.search(
                    r"try again in ([0-9.]+)ms",
                    message,
                    re.IGNORECASE
                )

                if match:

                    suggested_wait = (
                        float(match.group(1)) / 1000
                    )

                    wait_time = max(
                        2.0,
                        suggested_wait + 1.0
                    )

                else:

                    wait_time = 10.0


                print(
                    f"\n  Rate limit reached."
                    f" Waiting {wait_time:.1f}s..."
                )

                time.sleep(wait_time)


            except Exception as e:

                print(
                    "\n  ERROR:"
                )
                print(e)
                raise


        # --------------------------------------------
        # CONVERT WINNER SLOT → MODEL NAME
        # --------------------------------------------

        if winner_slot == "1":

            winner = m1

        elif winner_slot == "2":

            winner = m2

        else:

            winner = None
            unparsed_count += 1


        # --------------------------------------------
        # SAVE RESULT IN MEMORY
        # --------------------------------------------

        tournament_log.append({

            "english": english,

            "model_1": m1,

            "model_2": m2,

            "winner": winner,

            "winner_slot": winner_slot,

            "reason": reason,

        })


        completed_keys.add(key)

        completed += 1


        # --------------------------------------------
        # UPDATE WIN COUNT
        # --------------------------------------------

        if winner is not None:

            win_counts[winner] += 1


        # --------------------------------------------
        # SAVE IMMEDIATELY TO DRIVE
        # --------------------------------------------

        save_tournament_checkpoint()


        # --------------------------------------------
        # PROGRESS
        # --------------------------------------------

        print(
            f"  [{completed}/{total_matchups}] "
            f"{m1} vs {m2} "
            f"-> {winner}"
        )

        print(
            f"  ✓ Saved to Drive"
        )


        # Small pause between requests
        time.sleep(1.0)


# ------------------------------------------------------------
# 11. FINAL SUMMARY
# ------------------------------------------------------------

print()
print("=" * 60)
print("TOURNAMENT COMPLETE")
print("=" * 60)

print(
    f"Completed: {completed}/{total_matchups}"
)

print("\nWin counts:")

for name, count in sorted(
    win_counts.items(),
    key=lambda x: x[1],
    reverse=True
):

    print(
        f"  {name}: {count}"
    )

print(
    f"\nUnparsed judgments: {unparsed_count}"
)

print(
    f"\nCheckpoint saved at:"
)

print(checkpoint_file)

print("=" * 60)

Checkpoint file:
/content/drive/MyDrive/telugu-project/tournament_checkpoint.json

GROQ TOURNAMENT
Models: ['sft', 'dpo', 'simpo', 'orpo']
Pairs per example: 6
Held-out examples: 60
Total matchups: 360
Already completed: 8
Remaining: 352

Example 1/60
  SKIP: sft vs dpo (already saved)
  SKIP: sft vs simpo (already saved)
  SKIP: sft vs orpo (already saved)
  SKIP: dpo vs simpo (already saved)
  SKIP: dpo vs orpo (already saved)
  SKIP: simpo vs orpo (already saved)

Example 2/60
  SKIP: sft vs dpo (already saved)
  SKIP: sft vs simpo (already saved)
  [9/360] sft vs orpo -> sft
  ✓ Saved to Drive
  [10/360] dpo vs simpo -> dpo
  ✓ Saved to Drive
  [11/360] dpo vs orpo -> dpo
  ✓ Saved to Drive
  [12/360] simpo vs orpo -> simpo
  ✓ Saved to Drive

Example 3/60
  [13/360] sft vs dpo -> sft
  ✓ Saved to Drive
  [14/360] sft vs simpo -> sft
  ✓ Saved to Drive
  [15/360] sft vs orpo -> sft
  ✓ Saved to Drive
  [16/360] dpo vs simpo -> dpo
  ✓ Saved to Drive
  [17/360] dpo vs orpo -> dpo
  

## 27. Alignment-tax check

Confirms none of the three preference-optimization methods degraded basic translation ability on plain, generic sentences outside the training domain.

In [77]:
# ============================================================
# SECTION 27 — ALIGNMENT / TOURNAMENT ANALYSIS
# Convert IndicTrans internal output to actual Telugu script
# ============================================================

# Fresh processor for inference/post-processing
alignment_ip = IndicProcessor(inference=True)


def translate_for_alignment(model, sentence):
    # --------------------------------------------------------
    # 1. Prepare English source
    # --------------------------------------------------------
    src = alignment_ip.preprocess_batch(
        [sentence],
        src_lang="eng_Latn",
        tgt_lang="tel_Telu"
    )

    # --------------------------------------------------------
    # 2. Tokenize
    # --------------------------------------------------------
    inputs = tokenizer(
        src,
        return_tensors="pt",
        truncation=True,
        max_length=256
    )

    inputs = {
        k: v.to(model.device)
        for k, v in inputs.items()
    }

    # --------------------------------------------------------
    # 3. Generate
    # --------------------------------------------------------
    with torch.inference_mode():

        output = model.generate(
            **inputs,
            num_beams=1,
            max_new_tokens=128,
            use_cache=True
        )

    print("Generation finished.")

    # --------------------------------------------------------
    # 4. Decode using TARGET tokenizer
    # --------------------------------------------------------
    tokenizer._switch_to_target_mode()

    decoded = tokenizer.batch_decode(
        output,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True
    )

    tokenizer._switch_to_input_mode()

    # --------------------------------------------------------
    # 5. Convert IndicTrans internal representation
    #    → actual Telugu Unicode script
    # --------------------------------------------------------
    telugu = alignment_ip.postprocess_batch(
        decoded,
        lang="tel_Telu"
    )[0]

    return telugu


# ============================================================
# RUN ON HELD-OUT DATA
# ============================================================

alignment_results = []

for i, item in enumerate(heldout_data):

    english = extract_english(item["prompt"])

    print()
    print("=" * 70)
    print(f"Example {i + 1}/{len(heldout_data)}")
    print("=" * 70)

    print("English:", english)

    result = {
        "english": english,
        "reference": item.get("reference_telugu", "")
    }

    # --------------------------------------------------------
    # Generate Telugu from each model
    # --------------------------------------------------------

    for name, model in eval_models.items():

        print(f"\n{name.upper()}:")

        telugu = translate_for_alignment(
            model,
            english
        )

        result[name] = telugu

        print("Telugu:", telugu)

    alignment_results.append(result)


# ============================================================
# FINAL OUTPUT
# ============================================================

print()
print("=" * 70)
print("SECTION 27 COMPLETE")
print("=" * 70)

for i, result in enumerate(alignment_results):

    print(f"\nExample {i + 1}")
    print("English :", result["english"])
    print("Reference:", result["reference"])

    for name in eval_models.keys():
        print(f"{name.upper():8}:", result[name])


Example 1/60
English: The Maestro Edge 125 is also equipped with Hero MotoCorp's i3S (idle-Start-Stop-System) technology, which helps improve the fuel efficiency

SFT:
Generation finished.
Telugu: మాస్ట్రో ఎడ్జ్ 125 లో హీరో మోటోకార్ప్ యొక్క ఐ3ఎస్ (ఐడిల్-స్టార్ట్-స్టాప్-సిస్టమ్) టెక్నాలజీ కూడా ఉంది, ఇది ఇంధన సామర్థ్యాన్ని మెరుగుపరచడంలో సహాయపడుతుంది

DPO:
Generation finished.
Telugu: మాస్ట్రో ఎడ్జ్ 125 లో హీరో మోటోకార్ప్ యొక్క ఐ3ఎస్ (ఐడిల్-స్టార్ట్-స్టాప్-సిస్టమ్) టెక్నాలజీ కూడా ఉంది, ఇది ఇంధన సామర్థ్యాన్ని మెరుగుపరచడంలో సహాయపడుతుంది

SIMPO:
Generation finished.
Telugu: మాస్ట్రో ఎడ్జ్ 125లో హీరో మోటోకార్ప్ యొక్క ఐ3ఎస్ (ఐడిల్-స్టార్ట్-స్టాప్-సిస్టమ్) టెక్నాలజీ కూడా ఉంది, ఇది ఇంధన సామర్థ్యాన్ని మెరుగుపరచడంలో సహాయపడుతుంది

ORPO:
Generation finished.
Telugu: మాస్ట్రో ఎడ్జ్ 125లో హీరో మోటోకార్ప్ యొక్క ఐ3ఎస్ (ఐడిల్-స్టార్ట్-స్టాప్-సిస్టమ్) టెక్నాలజీ కూడా ఉంది, ఇది ఇంధన సామర్థ్యాన్ని మెరుగుపరచడంలో సహాయపడుతుంది

Example 2/60
English: is the question plaguing many.

SFT:
Generation finished.
Tel

## 28. Save final evaluation results

This is the file the write-up (GitHub README / Medium / LinkedIn post) draws from: win-rate table, held-out translations from all four models, the full tournament log, and the alignment-tax check.

In [78]:
final_eval_file = os.path.join(PROJECT_PATH, "final_eval_results.json")

with open(final_eval_file, "w", encoding="utf-8") as f:
    json.dump({
        "win_counts": win_counts,
        "heldout_translations": eval_results,
        "tournament_log": tournament_log,
        "alignment_tax_check": alignment_tax_results,
    }, f, ensure_ascii=False, indent=2)

print("Saved:", final_eval_file)
print("\nFinal win counts:", win_counts)


Saved: /content/drive/MyDrive/telugu-project/final_eval_results.json

Final win counts: {'sft': 134, 'dpo': 106, 'simpo': 65, 'orpo': 55}


## Output files

### Existing - reused, not regenerated
- `telugu_candidates_raw.jsonl`
- `telugu_judged_raw.jsonl`
- `preference_pairs.jsonl`
- `telugu_preferences.jsonl`
- `telugu_train.jsonl`
- `telugu_heldout.jsonl`
- `sft_checkpoint/`
- `sft_adapter/`

### Created by this notebook
- `dpo_checkpoint/`, `dpo_adapter/`
- `simpo_checkpoint/`, `simpo_adapter/`
- `orpo_checkpoint/`, `orpo_adapter/`
- `final_eval_results.json` - win-rate tournament results, held-out translations from all four models, full pairwise judge log, and the alignment-tax check

### Notes
- SFT is loaded once and never retrained.
- DPO and SimPO both start from the existing `sft_adapter`. ORPO trains from the raw base model, since it combines imitation and preference learning into a single stage.
- The Groq API key is read from Colab Secrets (`GROQ_API_KEY`), not hardcoded, to avoid exposing it if this notebook is shared or committed to a repo.
- This notebook contains only one Google Drive mount cell and never deletes the project folder.
- Encoder-decoder support (`is_encoder_decoder=True`) in `DPOTrainer`/`CPOTrainer`/`ORPOTrainer` is a less common path than the standard decoder-only setup these libraries are mostly tested around - if any training cell raises a shape or config error, that is the first thing to check.
